# Jack The Learner - Colab Training

**A humanoid robot brain with 105M parameters - ONE unified architecture.**

## Setup
1. **Runtime → Change runtime type → T4 GPU** (or A100 for faster)
2. Run cells in order
3. Checkpoints save locally during training (fast!)
4. Auto-backup to Drive every 100 epochs (safe!)

## Training Pipeline
```
Phase 0 (15-30 min)   Phase 1 (4-12 hrs)    Phase 2 (2-4 hrs)
Learn Physics    →    Learn Walking     →    Learn from Demos
SymPy ground truth    RL with safeguards    Imitation learning
+ EWC Fisher          + Replay buffer       + All safeguards
```

---

## 1. Setup Environment

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/JackTheLearner/checkpoints'
os.makedirs(DRIVE_PATH, exist_ok=True)
print('Google Drive mounted!')
print(f'Checkpoints will be backed up to: {DRIVE_PATH}')

In [ ]:
# Install dependencies
!pip install -q torch torchvision
!pip install -q sympy tqdm
print('Dependencies installed!')

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Clone repository
%cd /content
!rm -rf JackTheLearner 2>/dev/null
!git clone https://github.com/JannoLouwrens/JackTheLearner.git
%cd JackTheLearner

# Create local checkpoints dir
!mkdir -p checkpoints

# Copy any existing checkpoints FROM Drive (for resume)
!cp /content/drive/MyDrive/JackTheLearner/checkpoints/*.pt checkpoints/ 2>/dev/null || echo 'No existing checkpoints on Drive'

print('\nSetup complete!')
print('- Training saves to LOCAL /content/ (fast)')
print('- Auto-backup to Drive every 100 epochs (safe)')

In [ ]:
# Enable auto-backup in RobustTrainer config
# This patches the config to backup every 100 epochs

config_patch = '''
# Colab auto-backup config
import sys
sys.path.insert(0, '/content/JackTheLearner')

from RobustTrainer import RobustTrainerConfig

# Enable periodic Drive backups
RobustTrainerConfig.colab_backup_enabled = True
RobustTrainerConfig.colab_backup_interval = 100  # every 100 epochs
RobustTrainerConfig.colab_drive_path = '/content/drive/MyDrive/JackTheLearner/checkpoints'

print('Auto-backup enabled: every 100 epochs to Google Drive')
'''

exec(config_patch)

---
## 2. Phase 0: Learn Physics (15-30 min on GPU)

UnifiedBrain learns F=ma, torque, energy, momentum from SymPy ground truth.

**Auto-backup:** every 100 epochs to Google Drive

**Safeguards prepared:**
- EWC Fisher information computed (protects physics knowledge)
- Replay buffer saved (mix into later phases)

In [ ]:
# Quick test (2-3 min)
!python -c "
from RobustTrainer import RobustTrainer, RobustTrainerConfig
config = RobustTrainerConfig()
config.colab_backup_enabled = True
config.colab_backup_interval = 100
config.colab_drive_path = '/content/drive/MyDrive/JackTheLearner/checkpoints'
trainer = RobustTrainer(config)
trainer.train_phase0(num_epochs=5)
"

In [ ]:
# Full Phase 0 (15-30 min on T4 GPU)
# Auto-backs up to Drive every 100 epochs
!python -c "
from RobustTrainer import RobustTrainer, RobustTrainerConfig
config = RobustTrainerConfig()
config.colab_backup_enabled = True
config.colab_backup_interval = 100
config.colab_drive_path = '/content/drive/MyDrive/JackTheLearner/checkpoints'
trainer = RobustTrainer(config)
trainer.train_phase0(num_epochs=50)
print('Phase 0 complete! Physics knowledge learned.')
"

---
## 3. Phase 1: Learn Walking (4-12 hours)

RL training with **safeguards active**:
- 20% of each batch = Phase 0 physics data (replay buffer)
- EWC penalty protects physics weights
- Auto-backup every 100 epochs to Drive

In [ ]:
# Check Phase 0 checkpoint exists
!ls -la checkpoints/*.pt 2>/dev/null || echo 'No checkpoints yet - run Phase 0 first!'

In [ ]:
# Phase 1: RL Walking (requires gymnasium + mujoco)
!pip install -q gymnasium mujoco
!python -c "
from RobustTrainer import RobustTrainer, RobustTrainerConfig
config = RobustTrainerConfig()
config.colab_backup_enabled = True
config.colab_backup_interval = 100
config.colab_drive_path = '/content/drive/MyDrive/JackTheLearner/checkpoints'
trainer = RobustTrainer(config)
trainer.train_phase1(num_epochs=500)
print('Phase 1 complete! Robot can walk.')
"

---
## 4. Phase 2: Learn from Demos (2-4 hours)

Imitation learning with flow matching. Auto-backup every 100 epochs.

In [ ]:
# Phase 2: Imitation Learning
!python -c "
from RobustTrainer import RobustTrainer, RobustTrainerConfig
config = RobustTrainerConfig()
config.colab_backup_enabled = True
config.colab_backup_interval = 100
config.colab_drive_path = '/content/drive/MyDrive/JackTheLearner/checkpoints'
trainer = RobustTrainer(config)
trainer.train_phase2(num_epochs=100)
print('Phase 2 complete! Natural movement learned.')
"

---
## 5. Verify & Manual Backup

In [ ]:
# List all checkpoints
print('LOCAL checkpoints:')
!ls -lh checkpoints/
print('\nDRIVE checkpoints:')
!ls -lh /content/drive/MyDrive/JackTheLearner/checkpoints/

In [ ]:
# Manual backup (run anytime)
import shutil, glob, os
src = '/content/JackTheLearner/checkpoints'
dst = '/content/drive/MyDrive/JackTheLearner/checkpoints'
for f in glob.glob(f'{src}/*.pt'):
    shutil.copy2(f, dst)
    print(f'Copied {os.path.basename(f)}')
print('Backup complete!')

---
## Checkpoint Reference

| Phase | Checkpoint | Description |
|-------|------------|-------------|
| 0 | `phase0_best.pt` | Physics knowledge |
| 0 | `phase0_latest.pt` | Latest (for resume) |
| 0 | `ewc_state.pt` | Fisher information |
| 0 | `replay_buffer.pt` | Physics samples |
| 1 | `phase1_best.pt` | Best RL checkpoint |
| 1 | `phase1_latest.pt` | Latest (for resume) |
| 2 | `phase2_best.pt` | Final trained brain |

## Resume After Disconnect
1. Re-run setup cells (they copy checkpoints FROM Drive)
2. Training auto-resumes from `*_latest.pt`
3. Backups happen every 100 epochs automatically

---
**Model:** UnifiedBrain - 105M parameters (~420MB)  
**Author:** Janno Louwrens